In [ ]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Gemini API key loaded successfully.")

Gemini API key loaded successfully.


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini language model created successfully.")

Gemini language model created successfully.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Gemini embedding model created successfully.")

Gemini embedding model created successfully.


In [ ]:
loader = PyPDFLoader('/content/pdf_file.pdf')

documents = loader.load()

print("PDF loaded successfully.")
print("Number of pages:", len(documents))

PDF loaded successfully.
Number of pages: 1


In [ ]:
print("First page preview:")
print("=" * 60)

print(documents[0].page_content[:2000])

print("=" * 60)

print("Metadata:")
print(documents[0].metadata)

First page preview:
YOUSSEF REZGUI 
Youssefrezgui2004@gmail.com   |   +216 22 241 869   |   LinkedIn: Youssef Rezgui  
ABOUT ME 
Third-year student in Management Information Systems (Licence en Informatique de Gestion), specializing in 
Business Intelligence, at the Faculty of Economic Sciences and Management of Tunis (University of Tunis El 
Manar). Currently seeking a 4-month graduation internship, from February 2027 to May 2027, in the field of 
web development. 
I have solid foundations in HTML, CSS, JavaScript, PHP, Python, Java, C, SQL, and NoSQL. Passionate about web 
development, I am also self-learning modern libraries and frameworks to strengthen my technical skills and 
stay up to date with the most widely used technologies. 
Serious, motivated, curious, and analytically minded, I enjoy teamwork and adapt quickly to new 
environments. My goal is to join a company where I can apply my knowledge, contribute to real projects, and 
continue developing my professional skills. 
ED

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Document splitting completed.")
print("Number of chunks:", len(chunks))

Document splitting completed.
Number of chunks: 3


In [ ]:
print("First chunk:")
print("=" * 60)

print(chunks[0].page_content)

print("=" * 60)

print("Chunk metadata:")
print(chunks[0].metadata)

First chunk:
YOUSSEF REZGUI 
Youssefrezgui2004@gmail.com   |   +216 22 241 869   |   LinkedIn: Youssef Rezgui  
ABOUT ME 
Third-year student in Management Information Systems (Licence en Informatique de Gestion), specializing in 
Business Intelligence, at the Faculty of Economic Sciences and Management of Tunis (University of Tunis El 
Manar). Currently seeking a 4-month graduation internship, from February 2027 to May 2027, in the field of 
web development. 
I have solid foundations in HTML, CSS, JavaScript, PHP, Python, Java, C, SQL, and NoSQL. Passionate about web 
development, I am also self-learning modern libraries and frameworks to strengthen my technical skills and 
stay up to date with the most widely used technologies. 
Serious, motivated, curious, and analytically minded, I enjoy teamwork and adapt quickly to new 
environments. My goal is to join a company where I can apply my knowledge, contribute to real projects, and 
continue developing my professional skills. 
EDUCATION

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Gemini embedding model created successfully.")

Gemini embedding model created successfully.


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="document_chatbot",
    persist_directory="./chroma_db"
)

print("Documents successfully stored in ChromaDB.")

Documents successfully stored in ChromaDB.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Retriever created successfully.")

Retriever created successfully.


In [ ]:
test_question = input(
    "Enter a question about your document: "
)

retrieved_docs = retriever.invoke(test_question)

print("\nNumber of retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):

    print("\n" + "=" * 60)
    print(f"RETRIEVED CHUNK {i}")
    print("=" * 60)

    print(doc.page_content[:1000])

Enter a question about your document: how many pages

Number of retrieved chunks: 3

RETRIEVED CHUNK 1
YOUSSEF REZGUI 
Youssefrezgui2004@gmail.com   |   +216 22 241 869   |   LinkedIn: Youssef Rezgui  
ABOUT ME 
Third-year student in Management Information Systems (Licence en Informatique de Gestion), specializing in 
Business Intelligence, at the Faculty of Economic Sciences and Management of Tunis (University of Tunis El 
Manar). Currently seeking a 4-month graduation internship, from February 2027 to May 2027, in the field of 
web development. 
I have solid foundations in HTML, CSS, JavaScript, PHP, Python, Java, C, SQL, and NoSQL. Passionate about web 
development, I am also self-learning modern libraries and frameworks to strengthen my technical skills and 
stay up to date with the most widely used technologies. 
Serious, motivated, curious, and analytically minded, I enjoy teamwork and adapt quickly to new 
environments. My goal is to join a company where I can apply my knowledge

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
You are a document-based AI assistant.

Your job is to answer the user's question using ONLY
the information provided in the context.

Rules:
1. Use the provided context to answer.
2. Do not invent information.
3. Do not use outside knowledge.
4. If the answer is not present in the context,
   say that you could not find the answer in the document.
5. Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""
)

print("RAG prompt created successfully.")

RAG prompt created successfully.


In [ ]:
def format_documents(docs):
    """
    Combine retrieved document chunks into one context string.
    """

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


def rag_answer(question):
    """
    Retrieve relevant chunks and ask Gemini
    to answer using those chunks.
    """
    retrieved_docs = retriever.invoke(question)
    context = format_documents(retrieved_docs)
    formatted_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    response = llm.invoke(formatted_prompt)

    return response.content


print("RAG system created successfully.")

RAG system created successfully.


In [ ]:
question = input(
    "Ask a question about your document: "
)

answer = rag_answer(question)

print("\n" + "=" * 60)
print("QUESTION")
print("=" * 60)

print(question)

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)

print(answer)

Ask a question about your document: what is the document about


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



QUESTION
what is the document about

ANSWER
[{'type': 'text', 'text': "The document is a resume (CV) for Youssef Rezgui. It details his contact information, educational background (a Bachelor's Degree in Management Information Systems specializing in Business Intelligence), personal profile and career objective (seeking a 4-month graduation internship in web development), language proficiencies, extracurricular experience as a Human Resources Officer, and personal skills.", 'extras': {'signature': 'Es4OCssOARFNMg+bjoc0mKQUhq5r5Oy2CthLtaW2Z1nqSo1Gkv0UGXL3Ju8D24muaoFceP9gLdy85uLuahrLhop+Vp/cn653oamHExeZPfGjMPVKT9MY0SVH+olYQ6WsZY1Bp1xiMSnKSUGkeCkUaWvqJ1gnqUq+nFqMjzuO8Ks5YgkxDVUmyYqCKrvxWS0ayrvVW4SIs5DN26KJTYR6NvkLz3ZDgrVfzejcHe7+8qtXm5t7FNw8heAY+2KGg9SdYW2LB59/l7/6edPZYHyyHwvJUM1fC02dJgNfWesTS556NfEcmHbGHNolTrXh+WXCYkifF+ccyGKLH4KdTV9yfxEL7OleFfySj7dD5yNdlt4c68NJwohFcVReXIUQva+NAjcxgFMRX6WN0dQ+yFEBYt99Uilt+gzS65JSFQAvSF0FReQA3Tlzq0bHTgVZBtJ694t10aG+Sk4XTnQqLiq5aMaAsPkV2QiR9qjqFX4fluncMbM

In [ ]:
print("=" * 60)
print("DOCUMENT CHATBOT WITH RAG")
print("=" * 60)

print("The chatbot is ready!")
print("Ask questions about your uploaded document.")
print("Type 'exit' to stop.\n")


while True:

    question = input("You: ")
    if question.lower().strip() in [
        "exit",
        "quit",
        "stop"
    ]:
        print("\nChatbot stopped.")
        break
    if not question.strip():
        print("Please enter a question.\n")
        continue
    answer = rag_answer(question)

    print("\nAssistant:")
    print(answer)

    print("\n" + "-" * 60 + "\n")

DOCUMENT CHATBOT WITH RAG
The chatbot is ready!
Ask questions about your uploaded document.
Type 'exit' to stop.

You: does he have email


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Assistant:
[{'type': 'text', 'text': 'Yes, he has an email address, which is Youssefrezgui2004@gmail.com.', 'extras': {'signature': 'EqcFCqQFARFNMg/KqN6+cwV5DqRliW0ZQSvPGP7gIv4q0I59swgGh6NQAsRAIqXSyJQ2doTVjaV7p0/qbvF6EXdZJ7sA4Jcy2fMKHhMQfzwZr6mImBK7TXxwYHslb9ORMncCEGE0pTGkpJkfeH5yVm9sdLIQt/AfSJ7yQUsTGa4HouXdceDNC+1jlvpDIJOnjENjJhX+xjdpasIqEuZ02j6KbWq24CLS0oQCws3nZnAsyOJKl8kqFhujzdg8bWzp6NKj0W5YJzd/uopxZoAmRB/BwFfWzYoEjbP9gruddpCLCpB8Iz5gYsnEQ4G9zeH6vNr0UvnSs1wThLAndUweLDh7/fSifjndcCQyEEg/lP+ydvZ/lxGnbWmqZfKk6MYshmN9xW72M/0FCfCTaFpIA04feTFrmnav4YN5a8uVqfKcPXxb4GfLjf0GSWdfEhdEhP2qDKo/PJwhn0FHXueo5D+kp0/LPiWEsldBaKdsapx1zsrYDWOKjZraLmMwdVyp0X13J0gYekiAbmI84EsnjpASUMbQaHiZ1FHuOmKX39Wijiw8KYXLxJ19EVfThhm3w1bH0wY8DekfVnsboDeqnJAYJqZUFGNsnpWcjyu3rznoBA1LfphxHwiWY2rYufVFViXytmn8kfWTZibLM93pau6mi8eZwGA9vsS/ouVlOsV4j1xACKqaBgHwf5gHKzGFIS3cU9+tVQI4VxEQDoOaDNVYOMsQotAb0zqAuyzWZ90KxRvs6tG010s+G3TzHLkNbDwHpguNHOH59DlyKp/Ilc3B8x5Cikx0dULZxA+ifFusTSzt9OPADJYB+ZrefHa/dGlrG1z/Y/q7apmfTijVyC1Yu9AYIjIBKi